In [80]:
import math
import csv

In [81]:
class NeuralNetwork:
#     Необходимо помнить, что hiddenWeights должен быть передан
#     [
#         [w1, w4, w7]
#         [w2, w5, w8]
#         [w3, w6, w9]
#     ]
#
    def __init__(self, activationFunc, deactivationFunc, hiddenWeights: list, outputWeights: list, requestedEra: int):
        self.requestedEra = requestedEra
        # Веса скрытых синапсов
        self.hiddenWeights = hiddenWeights
        # Веса выходных синапсов
        self.outputWeights = outputWeights
        self.activationFunc = activationFunc
        self.deactivationFunc = deactivationFunc


        self.totalMSE = 0.0


        self.E = 0.7     # Скорость обучения
        self.a = 0.3     # Момент

        self.dataset = []
        self.answersList = []

        # Выходные значения скрытых нейронов
        self.hiddenNeuronOutput = []

        self.hiddenWeightDeltas = []      # Дельты скрытых синапсов
        self.outputWeightDeltas = []      # Дельты выходных синапсов

    def readFromCsv(self, datasetPath: str):
        # Чтение датасета из csv
        with open(datasetPath, 'r') as csvDataset:
            csvDatasetReader = csv.reader(csvDataset, delimiter='\t')
            for row in csvDatasetReader:
                self.dataset.append([ float(row[0]), float(row[1]), float(row[2]) ])

    def train(self):
        for trainSet in self.dataset:
            print('------------------------------')
            print(f'trainSet = {trainSet}')

            hiddenNeuronInputsValues = self.__calculateHiddenInputsValues(trainSet)
            print(f'hiddenNeuronInputsValues = {hiddenNeuronInputsValues}')

            hiddenNeuronOutputsValues = self.__calculateHiddenOutputsValues(hiddenNeuronInputsValues)
            print(f'hiddenNeuronOutputsValues = {hiddenNeuronOutputsValues}')

            # Поиск идеального значения
            ideal = self.__calcDiscriminant(trainSet[0], trainSet[1], trainSet[2])
            print(f'ideal = {ideal}')
            output = self.__calculateOutput(hiddenNeuronOutputsValues)
            print(f'real = {output}')

            # Запись ответа для расчёта MSE
            self.answersList.append(output)
            self.totalMSE = self.__calculateMSE(ideal, output)
            print(f'totalMSE = {self.totalMSE}')

            dO = (ideal - output)*self.deactivationFunc(output)
            print(f'dO = {dO}')

            dH = []
            for i in range(len(self.outputWeights)):
                dH.append( self.deactivationFunc(hiddenNeuronOutputsValues[i])*dO*self.outputWeights[i] )
            print(f'dH = {dH}')

            self.outputWeightDeltas = self.__calculateOutputWeightDeltas(dO, hiddenNeuronOutputsValues)
            print(f'outputDeltas = {self.outputWeightDeltas}')
            self.hiddenWeightDeltas = self.__calculateHiddenWeightDeltas(trainSet, dH)
            for deltas in self.hiddenWeightDeltas:
                print(f'hiddenDeltas = {deltas}')

            self.hiddenWeights = self.__calculateNewHiddenWeigths(self.hiddenWeights, self.hiddenWeightDeltas)
            for deltas in self.hiddenWeights:
                print(f'hiddenWeights = {deltas}')

            self.outputWeights = self.__calculateNewOutputWeigths(self.outputWeights, self.outputWeightDeltas)
            print(f'outputWeights = {self.outputWeights}')
            print('------------------------------')


    def __calcDiscriminant(self, a: float, b: float, c: float):
        return b**2 - 4*a*c

    def __calculateMSE(self, ideal:float, real: float):
        mse = float(0)
        for answer in self.answersList:
            mse += ((ideal - answer)**2)

        return mse / len(self.answersList)

    # Приходит [I_1, I_2, I_3]
    # Возвращается [H1_in, H2_in, H3_in]
    def __calculateHiddenInputsValues(self, trainSet: list) -> list:
        result = []
        for weights in self.hiddenWeights:
            halfResult = float(0)
            for i in range(len(weights)):
                halfResult += trainSet[i]*weights[i]

            result.append(halfResult)

        return result

    # Приходит [H1_in, H2_in, H3_in]
    # Возвращается [H1_out, H2_out, H3_out]
    def __calculateHiddenOutputsValues(self, inputs: list) -> list:
        result = []

        for inputValue in inputs:
            result.append(self.activationFunc(inputValue))

        return result

    # Приходит [H1_out, H2_out, H3_out]
    # Возвращается O_out
    def __calculateOutput(self, hiddenOutputsValues: list) -> float:
        result = float(0)
        for i in range(len(hiddenOutputsValues)):
            result += hiddenOutputsValues[i]*self.outputWeights[i]

        return result

    def __calculateOutputWeightDeltas(self, dO:float, hiddenNeuronOutputsValues: list) -> list:
        '''
        Расчёт дельт для весов выходных синапсов

        dO: float
        hiddenNeuronOutputsValues: list
        '''
        outputDeltas = []
        for i in range(len(hiddenNeuronOutputsValues)):
            outputGradient = dO*hiddenNeuronOutputsValues[i]

            previousOutputDelta = float(0)
            try:
                previousOutputDelta = float(self.outputWeightDeltas[i])
            except:
                pass

            delta = self.E*outputGradient + self.a*previousOutputDelta

            outputDeltas.append(delta)

        return outputDeltas


    def __calculateHiddenWeightDeltas(self, trainSet: list, dH: list) -> list:
        '''
        Расчёт дельт для весов скрытых синапсов

        trainSet: [a, b, c]
        dO: float
        output: float
        '''

        hiddenGradients = []
        for dH_n in dH:
            buffer = []
            for inputValue in trainSet:
                buffer.append( inputValue*dH_n )

            hiddenGradients.append(buffer)

        hiddenDeltas = []
        for i in range(len(hiddenGradients)):
            deltasOfLine = []
            for j in range (len(hiddenGradients[i])):
                hiddenGradient = hiddenGradients[i][j]
                previousHiddenDelta = float(0)
                try:
                    previousHiddenDelta = float(self.hiddenWeightDeltas[i])
                except:
                    pass

                delta = self.E*hiddenGradient + self.a*previousHiddenDelta

                deltasOfLine.append(delta)

            hiddenDeltas.append(deltasOfLine)

        return hiddenDeltas


    def __calculateNewHiddenWeigths(self, currentWeights: list, deltas: list):
        newWeights = []
        # Рачёт новых весов для скрытых синапсов
        for i in range(len(currentWeights)):
            lineOfNewWeights = []
            for j in range(len(currentWeights[i])):
                lineOfNewWeights.append( currentWeights[i][j] + deltas[i][j] )

            newWeights.append(lineOfNewWeights)

        return newWeights

    def __calculateNewOutputWeigths(self, currentWeights: list, deltas: list):
        newWeights = []
        # Рачёт новых весов для выходных синапсов
        for i in range(len(currentWeights)):
            newWeights.append( currentWeights[i] + deltas[i] )

        return newWeights



In [82]:
# Сигмоид
f_sig = lambda x: 1/(1 + math.exp(-1*x))
# Конечно-разностный аналог сигмоида для метода обр распределения
anti_f_sig = lambda x: (1 - x)*x

# Тангенсоид
f_tang = lambda x:(math.exp(2*x) - 1) / (math.exp(2*x) + 1)
# Конечно-разностный аналог Тангенсоид для метода обр распределения
f_anti_tang = lambda x: 1 - x**2

In [83]:
hiddenWeights = [[0.1, 0.9, 0.31], [0.4, 0.7, 0.11], [0.3, 0.2, 0.27]]
# hiddenWeights = [[1, 1, 6], [3, 7, 7], [5, 6, 0]]

outputWeights = [0.47, 0.51, 0.67]
# outputWeights = [1, 4, 6]

In [84]:
myAI = NeuralNetwork(f_sig, anti_f_sig, hiddenWeights, outputWeights, 1000)
# myAI = NeuralNetwork(f_tang, f_anti_tang, hiddenWeights, outputWeights, 1000)

In [85]:
myAI.readFromCsv("test_data.csv")

In [86]:
myAI.train()

------------------------------
trainSet = [3.0, 5.0, 2.0]
hiddenNeuronInputsValues = [5.42, 4.92, 2.44]
hiddenNeuronOutputsValues = [0.9955923665916371, 0.9927537604041685, 0.9198270878271877]
ideal = 1.0
real = 1.5905169789484113
totalMSE = 0.34871030242635837
dO = 0.5546296567433584
dH = [0.0011438997640297365, 0.002034827015022009, 0.02740386029182896]
outputDeltas = [0.3865295367773192, 0.38542747415454953, 0.3571143673893853]
hiddenDeltas = [0.0024021895044624465, 0.004003649174104077, 0.001601459669641631]
hiddenDeltas = [0.004273136731546219, 0.007121894552577031, 0.0028487578210308127]
hiddenDeltas = [0.057548106612840814, 0.09591351102140135, 0.03836540440856054]
hiddenWeights = [0.10240218950446245, 0.9040036491741041, 0.31160145966964164]
hiddenWeights = [0.40427313673154625, 0.707121894552577, 0.1128487578210308]
hiddenWeights = [0.3575481066128408, 0.29591351102140134, 0.30836540440856053]
outputWeights = [0.8565295367773191, 0.8954274741545496, 1.0271143673893852]
-------

OverflowError: (34, 'Numerical result out of range')

In [ ]:
print("myAI.totalMSE", myAI.totalMSE)
for weight in myAI.hiddenWeights:
    print(weight)
# print("myAI.hiddenWeights", myAI.hiddenWeights)
print("myAI.outputWeights", myAI.outputWeights)
print("myAI.answersList", myAI.answersList)

myAI.totalMSE 7.320008674890341e+155
[8297340.8454032345, 202375.08519574004, 2226116.1242343625]
[1.655617591684535e+35, 1.655617591684535e+35, 9.460671952483056e+34]
[1.441552394845105e+39, 1.441552394845105e+39, 8.237442256257744e+38]
myAI.outputWeights [14342324600725.777, 9.895841596195127e+73, 8.616346083571357e+77]
myAI.answersList [1.5905169789484113, 1.6501767565546541, 1.962295043622206, 2.0524215346572845, 2.079459481967808, 2.0875708661609647, 2.0900042814189117, 2.090734305996296, 2.0909533133695115, 2.091019015581476, 2.091038726245065, 2.091044639444142, 2.0910464134038653, 2.091046945591782, 2.0910471052481574, 2.0910471531450696, 2.0910471675141435, 2.0910471718248655, 2.0910471731179565, 2.091101408604994, 2.0911176792510675, 2.0911225604448895, 2.091124024803036, 2.0911020136597758, 71.64701208314212, 92.51377836885159, 98.77380825456443, 100.65181722027828, 101.21521990999244, 101.38424071690669, 101.43494695898096, 101.45015883160325, 101.45472239338993, 101.456091